In [ ]:
import sys
import os
import glob
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.data import Batch
from typing import List, Dict, Any, Optional, Tuple
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pathlib

# Add project root to path
PROJECT_ROOT = "/mnt/c/Users/obbee/research/notebooks/ML"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    os.environ["PYTHONPATH"] = PROJECT_ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")

from graph_data.utils import GraphGroupDataset, GraphRecord, NUM_GROUP_CLASSES, BIT_TO_CLASS
from graph_data.models import OrthogonalEndpointRegressor, GroupClassifier
from graph_data.plotting import plot_endpoint_error_distributions, plot_pull_distributions, plot_event_display
from graph_data.event_mixer import EventMixer, save_mixed_events
from pipelines.upstream_pipeline import UpstreamPipeline

/home/omar/miniconda3/envs/research/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_dir = "/mnt/c/Users/obbee/research/notebooks/ML/npy_data"
output_path = "/mnt/c/Users/obbee/research/notebooks/ML/processed/"

classifier_path = "/mnt/c/Users/obbee/research/notebooks/ML/model_weights/group_classifier_12_15_2025.pth" 
endpoint_path = "/mnt/c/Users/obbee/research/notebooks/ML/model_weights/endpoint_finder_12_25_2025.pth"

# 1. Load Data
print("Initializing EventMixer...")
hits_files = glob.glob(os.path.join(data_dir, "hits_batch_*.npy"))
info_files = glob.glob(os.path.join(data_dir, "group_info_batch_*.npy"))

if not hits_files:
    print("No hit files found. Exiting.")

Initializing EventMixer...


In [3]:
mixer = EventMixer(hits_files, info_files, max_events=200000)

Loading 289 file pairs...
Loading hits_batch_0.npy and group_info_batch_0.npy
Loading hits_batch_1.npy and group_info_batch_1.npy
Loading hits_batch_10.npy and group_info_batch_10.npy
Loading hits_batch_100.npy and group_info_batch_100.npy
Loading hits_batch_101.npy and group_info_batch_101.npy
Loading hits_batch_102.npy and group_info_batch_102.npy
Loading hits_batch_103.npy and group_info_batch_103.npy
Loading hits_batch_104.npy and group_info_batch_104.npy
Loading hits_batch_105.npy and group_info_batch_105.npy
Loading hits_batch_106.npy and group_info_batch_106.npy
Loading hits_batch_107.npy and group_info_batch_107.npy
Loading hits_batch_108.npy and group_info_batch_108.npy
Loading hits_batch_109.npy and group_info_batch_109.npy
Loading hits_batch_11.npy and group_info_batch_11.npy
Loading hits_batch_110.npy and group_info_batch_110.npy
Loading hits_batch_111.npy and group_info_batch_111.npy
Loading hits_batch_112.npy and group_info_batch_112.npy
Loading hits_batch_113.npy and gro

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pipeline = UpstreamPipeline(device)

In [5]:
classifier_args = {
    "num_classes": 3,
    "hidden": 150,      
    "num_blocks": 3,    
    "heads": 5,
    "dropout": 0.05
}
endpoint_args = {
    "in_channels": 4,
    "hidden": 150,
    "heads": 5,
    "layers": 2,
    "prob_dimension": 3,
    "dropout": 0.05
}

pipeline.load_models(
    classifier_path=classifier_path, 
    endpoint_path=endpoint_path,
    classifier_config=classifier_args,  
    endpoint_config=endpoint_args      
)

Loading classifier from /mnt/c/Users/obbee/research/notebooks/ML/model_weights/group_classifier_12_15_2025.pth
Loading endpoint regressor from /mnt/c/Users/obbee/research/notebooks/ML/model_weights/endpoint_finder_12_25_2025.pth


In [6]:
pipeline.process_unmixed_events(mixer.events)

Processing 588043 time groups from 200000 events in batches of 200...
Processed 2000/588043 groups
Processed 4000/588043 groups
Processed 6000/588043 groups
Processed 8000/588043 groups
Processed 10000/588043 groups
Processed 12000/588043 groups
Processed 14000/588043 groups
Processed 16000/588043 groups
Processed 18000/588043 groups
Processed 20000/588043 groups
Processed 22000/588043 groups
Processed 24000/588043 groups
Processed 26000/588043 groups
Processed 28000/588043 groups
Processed 30000/588043 groups
Processed 32000/588043 groups
Processed 34000/588043 groups
Processed 36000/588043 groups
Processed 38000/588043 groups
Processed 40000/588043 groups
Processed 42000/588043 groups
Processed 44000/588043 groups
Processed 46000/588043 groups
Processed 48000/588043 groups
Processed 50000/588043 groups
Processed 52000/588043 groups
Processed 54000/588043 groups
Processed 56000/588043 groups
Processed 58000/588043 groups
Processed 60000/588043 groups
Processed 62000/588043 groups
Proc

In [7]:
chunk_size = 10000
total_events = 200000
for i in range(0, total_events, chunk_size):
    # Generate a smaller chunk
    events_chunk = mixer.generate_mixed_batch(chunk_size)
    
    # Save to a separate file
    save_mixed_events(events_chunk, output_path + f"mixed_chunk_{i}_2nd.pt")
    
    # Free usage memory
    del events_chunk

Flattening data...
Concatenating tensors...
Saving to /mnt/c/Users/obbee/research/notebooks/ML/processed/mixed_chunk_0_2nd.pt...
Done.
Flattening data...
Concatenating tensors...
Saving to /mnt/c/Users/obbee/research/notebooks/ML/processed/mixed_chunk_10000_2nd.pt...
Done.
Flattening data...
Concatenating tensors...
Saving to /mnt/c/Users/obbee/research/notebooks/ML/processed/mixed_chunk_20000_2nd.pt...
Done.
Flattening data...
Concatenating tensors...
Saving to /mnt/c/Users/obbee/research/notebooks/ML/processed/mixed_chunk_30000_2nd.pt...
Done.
Flattening data...
Concatenating tensors...
Saving to /mnt/c/Users/obbee/research/notebooks/ML/processed/mixed_chunk_40000_2nd.pt...
Done.
Flattening data...
Concatenating tensors...
Saving to /mnt/c/Users/obbee/research/notebooks/ML/processed/mixed_chunk_50000_2nd.pt...
Done.
Flattening data...
Concatenating tensors...
Saving to /mnt/c/Users/obbee/research/notebooks/ML/processed/mixed_chunk_60000_2nd.pt...
Done.
Flattening data...
Concatenatin